# Data processing

In [30]:
### Used libraries ###

import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [31]:
### Data connection ###

path = r"Excel test - Business Analytics.xlsx"
raw_data = pd.read_excel(path, sheet_name="raw_data")
raw_data.head(10)

,Market,Departure Season,Departure Year,Booking Month,Passenger Group,Destination Type,Channel,Bookings,AOV,GBV EUR
0,CZ,Summer,2002,Jul 2025,Single,Sun & Beach,Online,1,0.0,0
1,CZ,Summer,2002,Jul 2025,Single,Sun & Beach,Total,1,0.0,0
2,CZ,Summer,2002,Jul 2025,Single,Total,NaN,1,0.0,0
3,CZ,Summer,2002,Jul 2025,Total,NaN,NaN,1,0.0,0
4,CZ,Summer,2002,Total,NaN,NaN,NaN,1,0.0,0
5,CZ,Summer,2022,Jan 2024,Single,Sun & Beach,Total,0,0.0,0
6,CZ,Summer,2022,Jan 2024,Single,Total,NaN,0,0.0,0
7,CZ,Summer,2022,Jan 2024,Young Couple (<30),Hotel Only,Total,0,0.0,0
8,CZ,Summer,2022,Jan 2024,Young Couple (<30),Total,NaN,0,0.0,0
9,CZ,Summer,2022,Jan 2024,Total,NaN,NaN,0,0.0,0


In [32]:
### Column mapping ###

column_mapping = {
    "Market": "market",
    "Departure Season": "departure_season",
    "Departure Year": "departure_year",
    "Booking Month": "booking_month",
    "Passenger Group": "passenger_group",
    "Destination Type": "destination_type",
    "Channel": "channel",
    "Bookings": "bookings",
    "AOV": "aov",
    "GBV EUR": "gbv_eur",
}

raw_data = raw_data.rename(columns=column_mapping)
raw_data.head(10)

,market,departure_season,departure_year,booking_month,passenger_group,destination_type,channel,bookings,aov,gbv_eur
0,CZ,Summer,2002,Jul 2025,Single,Sun & Beach,Online,1,0.0,0
1,CZ,Summer,2002,Jul 2025,Single,Sun & Beach,Total,1,0.0,0
2,CZ,Summer,2002,Jul 2025,Single,Total,NaN,1,0.0,0
3,CZ,Summer,2002,Jul 2025,Total,NaN,NaN,1,0.0,0
4,CZ,Summer,2002,Total,NaN,NaN,NaN,1,0.0,0
5,CZ,Summer,2022,Jan 2024,Single,Sun & Beach,Total,0,0.0,0
6,CZ,Summer,2022,Jan 2024,Single,Total,NaN,0,0.0,0
7,CZ,Summer,2022,Jan 2024,Young Couple (<30),Hotel Only,Total,0,0.0,0
8,CZ,Summer,2022,Jan 2024,Young Couple (<30),Total,NaN,0,0.0,0
9,CZ,Summer,2022,Jan 2024,Total,NaN,NaN,0,0.0,0


In [33]:
raw_data.info()
raw_data.describe()

<class 'pandas.DataFrame'>
RangeIndex: 37175 entries, 0 to 37174
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   market            37175 non-null  str    
 1   departure_season  33914 non-null  str    
 2   departure_year    33914 non-null  object 
 3   booking_month     36582 non-null  str    
 4   passenger_group   37102 non-null  str    
 5   destination_type  35896 non-null  str    
 6   channel           29629 non-null  str    
 7   bookings          37175 non-null  int64  
 8   aov               37175 non-null  float64
 9   gbv_eur           37175 non-null  int64  
dtypes: float64(1), int64(2), object(1), str(6)
memory usage: 2.8+ MB


,bookings,aov,gbv_eur
count,37175.000000,37175.000000,3.717500e+04
mean,148.982677,1285.950856,3.108856e+05
std,4674.892056,1978.214178,9.634597e+06
min,0.000000,0.000000,0.000000e+00
25%,0.000000,0.000000,0.000000e+00
50%,1.000000,484.900000,9.330000e+02
75%,11.000000,1928.775000,2.619850e+04
max,692079.000000,46129.000000,1.444647e+09


In [34]:
raw_data.nunique()

market                  5
departure_season        3
departure_year         12
booking_month          70
passenger_group         8
destination_type        5
channel                 3
bookings             1236
aov                 16213
gbv_eur             15681
dtype: int64

In [35]:
### Checking values ###

# Limit
unique_limit = 20

# List of categorical values
for col in raw_data.columns:
    n_unique = raw_data[col].nunique()
    if n_unique <= unique_limit:
        print(f"--- {col} ({n_unique} unikátních hodnot) ---")
        print(raw_data[col].unique())
        print()

--- market (5 unikátních hodnot) ---
<StringArray>
['CZ', 'PL', 'HU', 'SK', 'Total']
Length: 5, dtype: str

--- departure_season (3 unikátních hodnot) ---
<StringArray>
['Summer', 'Winter', nan, 'Total']
Length: 4, dtype: str

--- departure_year (12 unikátních hodnot) ---
[2002 2022 2023 2026 2024 2025 2027 'Total' 2021 nan 2015 2020 2019]

--- passenger_group (8 unikátních hodnot) ---
<StringArray>
[                'Single',                  'Total',                      nan,
     'Young Couple (<30)', 'Mid-age Couple (30-55)',               'Families',
                 'Groups',    'Senior Couple (>55)',                  'Other']
Length: 9, dtype: str

--- destination_type (5 unikátních hodnot) ---
<StringArray>
['Sun & Beach', 'Total', nan, 'Hotel Only', 'Other', 'Exotic']
Length: 6, dtype: str

--- channel (3 unikátních hodnot) ---
<StringArray>
['Online', 'Total', nan, 'Offline']
Length: 4, dtype: str



In [36]:
### Removing redundant values ###

# Drop values with Total
drop_total = (raw_data == "Total").any(axis=1)
print(f"Rows to drop (Total): {drop_total.sum()}")
clear_data = raw_data[~drop_total].reset_index(drop=True)

# Drop NaN values
drop_na = clear_data.isnull().any(axis=1)
print(f"Rows to drop (NaN): {drop_na.sum()}")
clear_data = clear_data[~drop_na].reset_index(drop=True)

# Drop zero GBV
drop_zero = (clear_data == 0).any(axis=1)
print(f"Rows to drop (0): {drop_zero.sum()}")
clear_data = clear_data[~drop_zero].reset_index(drop=True)

Rows to drop (Total): 26471
Rows to drop (NaN): 31
Rows to drop (0): 6


In [37]:
# Splitting the Booking month column into month and year
split_cols = clear_data["booking_month"].str.split(" ", expand=True)

clear_data["booking_year"] = split_cols[1]

month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
clear_data["booking_month"] = pd.Categorical(split_cols[0], categories=month_order, ordered=True)

col = clear_data.pop("booking_year")
clear_data.insert(clear_data.columns.get_loc("booking_month") + 1, "booking_year", col)

In [38]:
### Data type optimization ###

# quantitative values
clear_data["market"] = clear_data["market"].astype("category")                       # 4 unique values
clear_data["departure_season"] = clear_data["departure_season"].astype("category")   # 2 unique values
clear_data["departure_year"] = clear_data["departure_year"].astype("int16")          # 2020-2027
clear_data["booking_month"] = clear_data["booking_month"].astype("category")         # 36 unique values
clear_data["booking_year"] = clear_data["booking_year"].astype("int16")              # 4 unique values
clear_data["passenger_group"] = clear_data["passenger_group"].astype("category")     # 7 unique values
clear_data["destination_type"] = clear_data["destination_type"].astype("category")   # 4 unique values
clear_data["channel"] = clear_data["channel"].astype("category")                     # 2 unique values

# numerical values
clear_data["bookings"] = clear_data["bookings"].astype("int32")   # max 6104
clear_data["aov"] = clear_data["aov"].astype("float32")           # max 46129.0
clear_data["gbv_eur"] = clear_data["gbv_eur"].astype("int64")     # max 15094659

clear_data.head(10)

,market,departure_season,departure_year,booking_month,booking_year,passenger_group,destination_type,channel,bookings,aov,gbv_eur
0,PL,Winter,2026,Aug,2025,Young Couple (<30),Exotic,Offline,1,46129.000000,46129
1,PL,Winter,2023,Nov,2024,Families,Exotic,Online,1,40778.000000,40778
2,PL,Winter,2026,May,2025,Groups,Exotic,Offline,4,32059.250000,128237
3,HU,Winter,2024,Aug,2023,Families,Exotic,Online,1,30911.000000,30911
4,CZ,Winter,2024,Jun,2024,Groups,Other,Offline,1,27951.000000,27951
5,SK,Winter,2024,May,2024,Groups,Exotic,Offline,1,25710.000000,25710
6,SK,Winter,2025,Jun,2024,Groups,Exotic,Offline,1,25695.000000,25695
7,SK,Summer,2025,Sep,2024,Families,Exotic,Offline,1,25245.000000,25245
8,PL,Summer,2024,Nov,2024,Groups,Sun & Beach,Offline,1,24454.000000,24454
9,PL,Summer,2025,Oct,2024,Groups,Exotic,Offline,3,23536.666016,70610


In [39]:
### Development of Booking and GBV data series by months ###

monthly_summary = clear_data.groupby(["booking_year", "booking_month"], observed=True).agg(
    bookings=("bookings", "sum"),
    gbv_eur=("gbv_eur", "sum")
).reset_index()

monthly_summary

,booking_year,booking_month,bookings,gbv_eur
0,2022,Dec,2,4057
1,2023,Jan,1,2361
2,2023,Feb,8,19245
3,2023,Mar,34,120405
4,2023,Apr,86,304676
5,2023,May,128,446766
6,2023,Jun,224,721303
7,2023,Jul,441,1146575
8,2023,Aug,1375,3170659
9,2023,Sep,2975,7199268


# One-pager overview

In [40]:
def format_eu(value, decimals=2):
    s = f"{value:,.{decimals}f}"
    return s.replace(",", " ").replace(".", ",")

def yoy_span(value, center=0):
    diff = value - center
    cls = "yoy-positive" if diff > 0 else "yoy-negative" if diff < 0 else "yoy-flat"
    sign = "+" if diff > 0 else ""
    return f'<span class="{cls}">({sign}{format_eu(value, 1)} %)</span>'

def _wrap(header_html, rows_html):
    return f'''<table class="onepager">
{header_html}
{rows_html}
</table>'''

# Main table (Market x Channel + Total)
def df_to_html_table(df):
    header = "  <tr><th>Market</th><th>Channel</th><th>Bookings</th><th>AOV</th><th>GBV</th></tr>"
    def row(r):
        cls = "total-row" if r["channel"] in ("Total", "") else ""
        return f'''  <tr class="{cls}">
    <td>{r['market']}</td><td>{r['channel']}</td>
    <td>{format_eu(r['bookings'], 0)} {yoy_span(r['bookings_yoy_pct'])}</td>
    <td>{format_eu(r['aov'], 0)} EUR {yoy_span(r['aov_yoy_pct'])}</td>
    <td>{format_eu(r['gbv_eur'], 0)} EUR {yoy_span(r['gbv_eur_yoy_pct'])}</td>
  </tr>'''
    return _wrap(header, "\n".join(row(r) for _, r in df.iterrows()))

# Trend block (month in row, metrics in columns)
def trend_to_html_table(df):
    header = "  <tr><th>Month</th><th>Bookings</th><th>AOV</th><th>GBV</th></tr>"
    def row(month, r):
        return f'''  <tr>
    <td>{month}</td>
    <td>{format_eu(r['bookings'], 0)} {yoy_span(r['bookings_yoy_pct'])}</td>
    <td>{format_eu(r['aov'], 0)} EUR {yoy_span(r['aov_yoy_pct'])}</td>
    <td>{format_eu(r['gbv_eur'], 0)} EUR {yoy_span(r['gbv_eur_yoy_pct'])}</td>
  </tr>'''
    return _wrap(header, "\n".join(row(m, r) for m, r in df.iterrows()))

# KPI row (values ​​only - label and period are fixed in index.html)
def kpi_to_line_html(s):
    def item(label, value, unit, yoy):
        return f'<span class="metric"><strong>{label}:</strong> {format_eu(value, 0)}{unit} {yoy_span(yoy)}</span>'
    items = [
        item("Bookings", s["bookings"], "", s["bookings_yoy_pct"]),
        item("AOV", s["aov"], " EUR", s["aov_yoy_pct"]),
        item("GBV", s["gbv_eur"], " EUR", s["gbv_eur_yoy_pct"]),
    ]
    return "".join(items)

# Growth Index (Market in a row, index centered at 100)
def growth_to_html_table(df):
    header = "  <tr><th>Market</th><th>Share of GBV</th><th>GBV YoY</th><th>Growth Index</th><th>Contribution</th></tr>"
    def row(market, r):
        return f'''  <tr>
    <td>{market}</td>
    <td>{format_eu(r['gbv_share_pct'], 1)} %</td>
    <td>{yoy_span(r['gbv_yoy_pct'])}</td>
    <td>{format_eu(r['growth_index'], 0)}</td>
    <td>{format_eu(r['contribution_pct'], 1)} %</td>
  </tr>'''
    return _wrap(header, "\n".join(row(m, r) for m, r in df.iterrows()))

In [41]:
### Year-on-year comparison of the last closed month ###

# Setting the months to compare
current_year = 2025
current_month = "Oct"
prior_year = 2024

current = clear_data[(clear_data.booking_year == current_year) & (clear_data.booking_month == current_month)]
prior = clear_data[(clear_data.booking_year == prior_year) & (clear_data.booking_month == current_month)]

print(f"Current month ended: {current_month} {current_year}")
print(f"Comparison month: {current_month} {prior_year}")

# Function to compare Market x Channel between two months
def compare_channel_period(current, prior):
    cur = current.groupby(["market", "channel"], observed=True).agg(bookings=("bookings", "sum"), gbv_eur=("gbv_eur", "sum")).reset_index()
    pri = prior.groupby(["market", "channel"], observed=True).agg(bookings=("bookings", "sum"), gbv_eur=("gbv_eur", "sum")).reset_index()
    merged = cur.merge(pri, on=["market", "channel"], suffixes=("", "_prior"))
    merged["aov"] = merged["gbv_eur"] / merged["bookings"]
    merged["aov_prior"] = merged["gbv_eur_prior"] / merged["bookings_prior"]
    for col in ["bookings", "gbv_eur", "aov"]:
        merged[f"{col}_yoy_pct"] = (merged[col] - merged[f"{col}_prior"]) / merged[f"{col}_prior"] * 100
    return merged
channel_level = compare_channel_period(current, prior)

# Market Total
market_level = channel_level.groupby("market", observed=True)[["bookings", "gbv_eur", "bookings_prior", "gbv_eur_prior"]].sum().reset_index()
market_level["aov"] = market_level["gbv_eur"] / market_level["bookings"]
market_level["aov_prior"] = market_level["gbv_eur_prior"] / market_level["bookings_prior"]
for col in ["bookings", "gbv_eur", "aov"]:
    market_level[f"{col}_yoy_pct"] = (market_level[col] - market_level[f"{col}_prior"]) / market_level[f"{col}_prior"] * 100
market_level["channel"] = "Total"

# Grand Total
grand_level = channel_level[["bookings", "gbv_eur", "bookings_prior", "gbv_eur_prior"]].sum().to_frame().T
grand_level["aov"] = grand_level["gbv_eur"] / grand_level["bookings"]
grand_level["aov_prior"] = grand_level["gbv_eur_prior"] / grand_level["bookings_prior"]
for col in ["bookings", "gbv_eur", "aov"]:
    grand_level[f"{col}_yoy_pct"] = (grand_level[col] - grand_level[f"{col}_prior"]) / grand_level[f"{col}_prior"] * 100
grand_level["market"] = "Grand Total"
grand_level["channel"] = ""

# Compiling the final table in the correct row order
rows = []
for m in ["CZ", "HU", "PL", "SK"]:
    rows.append(channel_level[channel_level.market == m])
    rows.append(market_level[market_level.market == m])
rows.append(grand_level)

final_table = pd.concat(rows, ignore_index=True)[["market", "channel", "bookings", "bookings_yoy_pct", "aov", "aov_yoy_pct", "gbv_eur", "gbv_eur_yoy_pct"]]

os.makedirs("output", exist_ok=True)
with open("output/main_table.html", "w", encoding="utf-8") as f:
    f.write(df_to_html_table(final_table))

final_table

Current month ended: Oct 2025
Comparison month: Oct 2024


,market,channel,bookings,bookings_yoy_pct,aov,aov_yoy_pct,gbv_eur,gbv_eur_yoy_pct
0,CZ,Offline,2387,10.000000,2279.712610,9.820157,5441674,20.802173
1,CZ,Online,6084,13.592233,2100.855687,5.774079,12781606,20.151138
2,CZ,Total,8471,12.556471,2151.254870,6.919493,18223280,20.344808
3,HU,Offline,776,15.133531,2865.244845,3.376095,2223430,19.020548
4,HU,Online,611,30.835118,2047.296236,2.730197,1250898,34.407174
5,HU,Total,1387,21.560035,2504.922855,2.119956,3474328,24.137054
6,PL,Offline,4572,17.080666,2250.456912,7.849655,10289089,26.271094
7,PL,Online,3796,-3.971667,1690.357745,6.291903,6416598,2.070343
8,PL,Total,8368,6.490201,1996.377510,8.678128,16705687,15.731557
9,SK,Offline,980,56.299841,2876.467347,12.148346,2818938,75.287686


In [42]:
### Developments over the last 6 months ###

# Months included
trend_months = ["May", "Jun", "Jul", "Aug", "Sep", "Oct"]

# Filtering data for the current and comparative year
trend_cur = clear_data[(clear_data.booking_year == current_year) & (clear_data.booking_month.isin(trend_months))]
trend_pri = clear_data[(clear_data.booking_year == prior_year) & (clear_data.booking_month.isin(trend_months))]

# Sum of bookings/GBV by month + calculation of AOV from these sums
trend_cur_agg = trend_cur.groupby("booking_month", observed=True).agg(bookings=("bookings", "sum"), gbv_eur=("gbv_eur", "sum")).reindex(trend_months)
trend_pri_agg = trend_pri.groupby("booking_month", observed=True).agg(bookings=("bookings", "sum"), gbv_eur=("gbv_eur", "sum")).reindex(trend_months)
trend_cur_agg["aov"] = trend_cur_agg.gbv_eur / trend_cur_agg.bookings
trend_pri_agg["aov"] = trend_pri_agg.gbv_eur / trend_pri_agg.bookings

# Compiling the final table
trend_table = pd.DataFrame(index=trend_months)
for metric in ["bookings", "aov", "gbv_eur"]:
    trend_table[metric] = trend_cur_agg[metric]
    trend_table[f"{metric}_yoy_pct"] = (trend_cur_agg[metric] - trend_pri_agg[metric]) / trend_pri_agg[metric] * 100

os.makedirs("output", exist_ok=True)
with open("output/trend_table.html", "w", encoding="utf-8") as f:
    f.write(trend_to_html_table(trend_table))

trend_table

,bookings,bookings_yoy_pct,aov,aov_yoy_pct,gbv_eur,gbv_eur_yoy_pct
May,30723,4.642371,2059.862155,3.127826,63285145,7.915401
Jun,38272,4.411404,2046.980168,0.728125,78342025,5.171649
Jul,46806,8.853694,2128.414370,6.094022,99622563,15.487262
Aug,39443,7.782484,1997.341556,8.594818,78781143,17.046192
Sep,29114,15.020544,1906.109260,6.391800,55494465,22.372427
Oct,20102,12.232706,2150.603074,8.161307,43231423,21.392362


In [43]:
# KPI summary
kpi_summary = grand_level[["bookings", "bookings_yoy_pct", "aov", "aov_yoy_pct", "gbv_eur", "gbv_eur_yoy_pct"]].iloc[0]

os.makedirs("output", exist_ok=True)
with open("output/kpi_table.html", "w", encoding="utf-8") as f:
    f.write(kpi_to_line_html(kpi_summary))

kpi_summary

bookings            2.010200e+04
bookings_yoy_pct    1.223271e+01
aov                 2.150603e+03
aov_yoy_pct         8.161307e+00
gbv_eur             4.323142e+07
gbv_eur_yoy_pct     2.139236e+01
Name: 0, dtype: float64

In [44]:
### Growth Index + Contribution to Growth ###

# Month selection
ytd_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct"]

# Filtering data for the current and comparative year
ytd_cur = clear_data[(clear_data.booking_year == current_year) & (clear_data.booking_month.isin(ytd_months))]
ytd_pri = clear_data[(clear_data.booking_year == prior_year) & (clear_data.booking_month.isin(ytd_months))]

# Sum of GBV by market
market_cur = ytd_cur.groupby("market", observed=True).agg(gbv_eur=("gbv_eur", "sum"))
market_pri = ytd_pri.groupby("market", observed=True).agg(gbv_eur=("gbv_eur", "sum"))

# Share of total GBV of the given market
growth_table = pd.DataFrame(index=market_cur.index)
growth_table["gbv_share_pct"] = market_cur.gbv_eur / market_cur.gbv_eur.sum() * 100

# Absolute and percentage year-on-year change in GBV of the given market
growth_table["gbv_delta"] = market_cur.gbv_eur - market_pri.gbv_eur
growth_table["gbv_yoy_pct"] = growth_table.gbv_delta / market_pri.gbv_eur * 100

# Growth Index - market pace versus average pace of the entire portfolio
portfolio_yoy_pct = growth_table.gbv_delta.sum() / market_pri.gbv_eur.sum() * 100
growth_table["growth_index"] = growth_table.gbv_yoy_pct / portfolio_yoy_pct * 100

# Contribution to Growth
growth_table["contribution_pct"] = growth_table.gbv_delta / growth_table.gbv_delta.sum() * 100

# Ranking by Growth Index
growth_table = growth_table.sort_values("growth_index", ascending=False)

os.makedirs("output", exist_ok=True)
with open("output/growth_table.html", "w", encoding="utf-8") as f:
    f.write(growth_to_html_table(growth_table))

growth_table

,gbv_share_pct,gbv_delta,gbv_yoy_pct,growth_index,contribution_pct
market,,,,,
SK,16.963920,14076655,14.011682,153.938543,24.989494
PL,29.832409,21256174,11.797699,129.614746,37.734890
CZ,43.556322,17646271,6.383263,70.129353,31.326433
HU,9.647349,3351192,5.423721,59.587402,5.949183


# Most impactful dimensions

## Determining the dimensions with the greatest impact

In [45]:
### AOV - Eta-squared ###

# Selecting dimensions for comparison
dimensions = ["passenger_group", "destination_type", "market", "channel", "departure_season"]

# Total weighted average AOV and total variance
grand_mean = (clear_data["aov"] * clear_data["bookings"]).sum() / clear_data["bookings"].sum()
ss_total = (clear_data["bookings"] * (clear_data["aov"] - grand_mean) ** 2).sum()

# Auxiliary column for calculating the weighted average in each group
clear_data["value_x_weight"] = clear_data["aov"] * clear_data["bookings"]

# Weighted average AOV by category and variance between categories
results = {}
for dim in dimensions:
    grouped = clear_data.groupby(dim, observed=True).agg(
        weight=("bookings", "sum"),
        value_x_weight=("value_x_weight", "sum")
    )
    grouped["mean"] = grouped["value_x_weight"] / grouped["weight"]
    ss_between = (grouped["weight"] * (grouped["mean"] - grand_mean) ** 2).sum()
    results[dim] = ss_between / ss_total * 100

# Compiling and sorting
impact_ranking = pd.Series(results, name="aov_variance_explained_pct").sort_values(ascending=False)
impact_ranking

passenger_group     39.142982
destination_type    30.247070
market               4.085976
channel              3.187940
departure_season     0.040443
Name: aov_variance_explained_pct, dtype: float64

In [46]:
### GBV- Normalized HHI (Herfindahl-Hirschman Index) ###

# Determining the uneven distribution of GBV across dimension categories
gbv_results = {}
for dim in dimensions:
    shares = clear_data.groupby(dim, observed=True)["gbv_eur"].sum()
    shares = shares / shares.sum()
    k = len(shares)
    hhi = (shares ** 2).sum()
    gbv_results[dim] = (hhi - 1 / k) / (1 - 1 / k) * 100

# Compiling and sorting
concentration_gbv = pd.Series(gbv_results, name="gbv_concentration").sort_values(ascending=False)
concentration_gbv

destination_type    61.897380
departure_season    36.272798
passenger_group     18.719790
market               9.086095
channel              1.283967
Name: gbv_concentration, dtype: float64

In [47]:
### Booking - Normalized HHI (Herfindahl-Hirschman Index) ###

# Determining the uneven distribution of Booking across dimension categories
bookings_results = {}
for dim in dimensions:
    shares = clear_data.groupby(dim, observed=True)["bookings"].sum()
    shares = shares / shares.sum()
    k = len(shares)
    hhi = (shares ** 2).sum()
    bookings_results[dim] = (hhi - 1 / k) / (1 - 1 / k) * 100

# Compiling and sorting
concentration_bookings = pd.Series(bookings_results, name="bookings_concentration").sort_values(ascending=False)
concentration_bookings

destination_type    66.228379
departure_season    37.158611
passenger_group     12.094970
market              11.665625
channel              3.749221
Name: bookings_concentration, dtype: float64

## Display of the most influential impact in graphs

In [48]:
### Dimension aggregation (Passenger Group, Destination Type, Departure Season) ###

# Passenger Group aggregation
passenger_group_agg = clear_data.groupby("passenger_group", observed=True).agg(bookings=("bookings", "sum"), gbv_eur=("gbv_eur", "sum"))
passenger_group_agg["aov"] = passenger_group_agg.gbv_eur / passenger_group_agg.bookings
passenger_group_agg["gbv_share_pct"] = passenger_group_agg.gbv_eur / passenger_group_agg.gbv_eur.sum() * 100
passenger_group_agg["bookings_share_pct"] = passenger_group_agg.bookings / passenger_group_agg.bookings.sum() * 100
passenger_group_agg["gbv_millions"] = passenger_group_agg["gbv_eur"] / 1_000_000
passenger_group_agg = passenger_group_agg.sort_values("aov", ascending=False)
passenger_group_share_sorted = passenger_group_agg.sort_values("gbv_share_pct", ascending=False)

# Destination Type aggregation
destination_type_agg = clear_data.groupby("destination_type", observed=True).agg(bookings=("bookings", "sum"), gbv_eur=("gbv_eur", "sum"))
destination_type_agg["aov"] = destination_type_agg.gbv_eur / destination_type_agg.bookings
destination_type_agg["gbv_share_pct"] = destination_type_agg.gbv_eur / destination_type_agg.gbv_eur.sum() * 100
destination_type_agg["gbv_millions"] = destination_type_agg["gbv_eur"] / 1_000_000
destination_type_agg = destination_type_agg.sort_values("aov", ascending=False)

# Departure Season aggregation
departure_season_agg = clear_data.groupby("departure_season", observed=True).agg(gbv_eur=("gbv_eur", "sum"))
departure_season_agg["share_pct"] = departure_season_agg.gbv_eur / departure_season_agg.gbv_eur.sum() * 100
departure_season_agg["gbv_millions"] = departure_season_agg["gbv_eur"] / 1_000_000

In [49]:
FONT_STACK = "Segoe UI, Roboto, Helvetica Neue, Arial, sans-serif"
sea_blue = "#05a8e6"
blue_dark = "#0c4a6e"
neutral_gray = "#94a3b8"

CHART_LAYOUT = dict(
    template="plotly_white",
    separators=", ",
    font=dict(family=FONT_STACK, size=13, color="#1f2937"),
    title_font=dict(size=16, color="#0f172a"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=60, r=40, t=60, b=90),
    xaxis=dict(showgrid=False, zeroline=False, showline=True, linecolor="#e2e8f0"),
    yaxis=dict(showgrid=True, gridcolor="#f1f5f9", zeroline=False, showline=False),
    hoverlabel=dict(bgcolor="white", font_size=13, font_family=FONT_STACK),
)

In [50]:
### Graf - Share of GBV by Passenger Group ###

fig_gbv_share_passenger_group = go.Figure()
fig_gbv_share_passenger_group.add_trace(go.Bar(
    x=passenger_group_share_sorted.index, y=passenger_group_share_sorted["gbv_share_pct"], marker_color=sea_blue,
    customdata=passenger_group_share_sorted[["gbv_millions", "bookings_share_pct", "bookings", "aov"]],
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Share of GBV: %{y:.1f} %<br>"
        "GBV: %{customdata[0]:.0f} million EUR<br>"
        "Share of Bookings: %{customdata[1]:.1f} %<br>"
        "Bookings: %{customdata[2]:,.0f}<br>"
        "AOV: %{customdata[3]:,.0f} EUR"
        "<extra></extra>"
    ),
))

fig_gbv_share_passenger_group.update_layout(xaxis_title="Passenger Group", yaxis_title="Share of GBV (%)")
fig_gbv_share_passenger_group.update_layout(**CHART_LAYOUT)
fig_gbv_share_passenger_group.update_xaxes(tickangle=-45)

os.makedirs("output", exist_ok=True)
fig_gbv_share_passenger_group.write_html("output/share_of_gbv_by_passenger_group.html", full_html=False, include_plotlyjs=False)

fig_gbv_share_passenger_group.show()

In [51]:
### Graf - AOV by Passenger Group ###

fig_aov_passenger_group = go.Figure()

fig_aov_passenger_group.add_trace(go.Bar(
    x=passenger_group_agg.index,
    y=passenger_group_agg["aov"],
    marker_color=sea_blue,
    customdata=passenger_group_agg[["bookings", "gbv_millions", "gbv_share_pct"]],
    hovertemplate=(
        "<b>%{x}</b><br>"
        "AOV: %{y:,.0f} EUR<br>"
        "Bookings: %{customdata[0]:,.0f}<br>"
        "GBV: %{customdata[1]:,.0f} million EUR (%{customdata[2]:.1f} % of total)"
        "<extra></extra>"
    ),
))

fig_aov_passenger_group.update_layout(xaxis_title="Passenger Group", yaxis_title="AOV (EUR)")
fig_aov_passenger_group.update_layout(**CHART_LAYOUT)
fig_aov_passenger_group.update_xaxes(tickangle=-45)
fig_aov_passenger_group.update_yaxes(tickformat=",.0f")

os.makedirs("output", exist_ok=True)
fig_aov_passenger_group.write_html("output/aov_by_passenger_group.html", full_html=False, include_plotlyjs=False)

fig_aov_passenger_group.show()

In [52]:
fig_aov_destination_type = go.Figure()
fig_aov_destination_type.add_trace(go.Bar(
    x=destination_type_agg.index, y=destination_type_agg["aov"], marker_color=sea_blue,
    customdata=destination_type_agg[["bookings", "gbv_millions", "gbv_share_pct"]],
    hovertemplate=(
        "<b>%{x}</b><br>"
        "AOV: %{y:,.0f} EUR<br>"
        "Bookings: %{customdata[0]:,.0f}<br>"
        "GBV: %{customdata[1]:,.0f} million EUR (%{customdata[2]:.1f} % of total)"
        "<extra></extra>"
    ),
))
fig_aov_destination_type.update_layout(xaxis_title="Destination Type", yaxis_title="AOV (EUR)")
fig_aov_destination_type.update_layout(**CHART_LAYOUT)
fig_aov_destination_type.update_yaxes(tickformat=",.0f")

os.makedirs("output", exist_ok=True)
fig_aov_destination_type.write_html("output/aov_by_destination_type.html", full_html=False, include_plotlyjs=False)

fig_aov_destination_type.show()

In [53]:
### Grafs - Bubble charts ###

per_person_note = {
    "Single": f'{passenger_group_agg.loc["Single","aov"]/1:.0f} EUR/person',
    "Young Couple (<30)": f'{passenger_group_agg.loc["Young Couple (<30)","aov"]/2:.0f} EUR/person',
    "Mid-age Couple (30-55)": f'{passenger_group_agg.loc["Mid-age Couple (30-55)","aov"]/2:.0f} EUR/person',
    "Senior Couple (>55)": f'{passenger_group_agg.loc["Senior Couple (>55)","aov"]/2:.0f} EUR/person',
    "Families": f'~{passenger_group_agg.loc["Families","aov"]/5:.0f}-{passenger_group_agg.loc["Families","aov"]/3:.0f} EUR/person (est., 3-5 ppl)',
    "Groups": f'~{passenger_group_agg.loc["Groups","aov"]/20:.0f}-{passenger_group_agg.loc["Groups","aov"]/5:.0f} EUR/person (est., 5-20 ppl)',
    "Other": "n/a (negligible volume)",
}
passenger_group_agg["per_person_note"] = passenger_group_agg.index.map(per_person_note)

# Bubble size setting
max_size_px = 70
sizeref_pg = 2 * passenger_group_agg["gbv_millions"].max() / (max_size_px ** 2)

# Bookings vs AOV by Destination Type
fig_bubble_passenger_group = go.Figure()
fig_bubble_passenger_group.add_trace(go.Scatter(
    x=passenger_group_agg["bookings"], y=passenger_group_agg["aov"], mode="markers+text",
    text=passenger_group_agg.index, textposition="middle center", textfont=dict(size=9),
    marker=dict(size=passenger_group_agg["gbv_millions"], sizemode="area", sizeref=sizeref_pg, sizemin=6, color=sea_blue, opacity=0.6, line=dict(width=0)),
    customdata=passenger_group_agg[["gbv_millions", "per_person_note"]],
    hovertemplate="<b>%{text}</b><br>Bookings: %{x:,.0f}<br>AOV per booking: %{y:,.0f} EUR<br>GBV: %{customdata[0]:,.0f} million EUR<br><i>Est. %{customdata[1]}</i><extra></extra>",
))
fig_bubble_passenger_group.update_layout(xaxis_title="Bookings", yaxis_title="AOV per Booking (EUR)")
fig_bubble_passenger_group.update_layout(**CHART_LAYOUT)
fig_bubble_passenger_group.update_yaxes(tickformat=",.0f")

os.makedirs("output", exist_ok=True)
fig_bubble_passenger_group.write_html("output/bubble_passenger_group.html", full_html=False, include_plotlyjs=False)

fig_bubble_passenger_group.show()

# Bubble size setting
sizeref_dt = 2 * destination_type_agg["gbv_millions"].max() / (max_size_px ** 2)

# Bookings vs AOV by Destination Type
fig_bubble_destination_type = go.Figure()
fig_bubble_destination_type.add_trace(go.Scatter(
    x=destination_type_agg["bookings"], y=destination_type_agg["aov"], mode="markers+text",
    text=destination_type_agg.index, textposition="middle center", textfont=dict(size=9),
    marker=dict(size=destination_type_agg["gbv_millions"], sizemode="area", sizeref=sizeref_dt, sizemin=6, color=sea_blue, opacity=0.6, line=dict(width=0)),
    customdata=destination_type_agg[["gbv_millions"]],
    hovertemplate="<b>%{text}</b><br>Bookings: %{x:,.0f}<br>AOV: %{y:,.0f} EUR<br>GBV: %{customdata[0]:,.0f} EUR<extra></extra>",
))
fig_bubble_destination_type.update_layout(xaxis_title="Bookings", yaxis_title="AOV (EUR)")
fig_bubble_destination_type.update_layout(**CHART_LAYOUT)
fig_bubble_destination_type.update_yaxes(tickformat=",.0f")
fig_bubble_destination_type.update_xaxes(tickformat=",.0f")

os.makedirs("output", exist_ok=True)
fig_bubble_destination_type.write_html("output/bubble_destination_type.html", full_html=False, include_plotlyjs=False)

fig_bubble_destination_type.show()

In [54]:
fig_gbv_departure_season = go.Figure()

fig_gbv_departure_season.add_trace(go.Bar(
    y=["GBV"], x=[departure_season_agg.loc["Summer","share_pct"]], name="Summer", orientation="h",
    marker_color=sea_blue, text=f'{format_eu(departure_season_agg.loc["Summer","share_pct"], 1)} %', textposition="inside", insidetextanchor="middle",
    textfont=dict(color="white"), customdata=[[departure_season_agg.loc["Summer","gbv_millions"]]],
    hovertemplate="<b>Summer</b><br>Share: %{x:.1f} %<br>GBV: %{customdata[0]:,.0f} million EUR<extra></extra>",
))
fig_gbv_departure_season.add_trace(go.Bar(
    y=["GBV"], x=[departure_season_agg.loc["Winter","share_pct"]], name="Winter", orientation="h",
    marker_color=blue_dark, text=f'{format_eu(departure_season_agg.loc["Winter","share_pct"], 1)} %', textposition="inside", insidetextanchor="middle",
    textfont=dict(color="white"), customdata=[[departure_season_agg.loc["Winter","gbv_millions"]]],
    hovertemplate="<b>Winter</b><br>Share: %{x:.1f} %<br>GBV: %{customdata[0]:,.0f} million EUR<extra></extra>",
))

fig_gbv_departure_season.update_layout(barmode="stack", height=280,
    legend=dict(orientation="h", yanchor="bottom", y=-0.35, xanchor="center", x=0.5))
fig_gbv_departure_season.update_layout(**CHART_LAYOUT)
fig_gbv_departure_season.update_xaxes(range=[0, 100], ticksuffix=" %", dtick=10, showgrid=True, gridcolor="#e2e8f0", zeroline=False, showline=True)
fig_gbv_departure_season.update_yaxes(visible=False)

os.makedirs("output", exist_ok=True)
fig_gbv_departure_season.write_html("output/gbv_departure_season.html", full_html=False, include_plotlyjs=False)

fig_gbv_departure_season.show()

In [55]:
s2024 = clear_data[clear_data.booking_year == 2024].groupby("booking_month", observed=True).agg(bookings=("bookings", "sum"), gbv_eur=("gbv_eur", "sum")).reindex(month_order)
s2024["aov"] = s2024.gbv_eur / s2024.bookings
s2024["gbv_millions"] = s2024.gbv_eur / 1_000_000

s2025 = clear_data[clear_data.booking_year == 2025].groupby("booking_month", observed=True).agg(bookings=("bookings", "sum"), gbv_eur=("gbv_eur", "sum")).reindex(month_order[:10])
s2025["aov"] = s2025.gbv_eur / s2025.bookings
s2025["gbv_millions"] = s2025.gbv_eur / 1_000_000

yoy_pct = (s2025["bookings"] - s2024["bookings"][:10]) / s2024["bookings"][:10] * 100

fig_seasonality = go.Figure()

fig_seasonality.add_trace(go.Scatter(
    x=s2024.index, y=s2024["bookings"], mode="lines+markers", name="2024",
    line=dict(color=blue_dark, width=2), marker=dict(size=7),
    customdata=s2024[["gbv_millions", "aov"]],
    hovertemplate="<b>%{x} 2024</b><br>Bookings: %{y:,.0f}<br>GBV: %{customdata[0]:,.0f} M EUR<br>AOV: %{customdata[1]:,.0f} EUR<extra></extra>",
))

fig_seasonality.add_trace(go.Scatter(
    x=s2025.index, y=s2025["bookings"], mode="lines+markers", name="2025",
    line=dict(color=sea_blue, width=2), marker=dict(size=7),
    customdata=pd.concat([s2025[["gbv_millions", "aov"]], yoy_pct.rename("yoy")], axis=1),
    hovertemplate="<b>%{x} 2025</b><br>Bookings: %{y:,.0f}<br>GBV: %{customdata[0]:,.0f} million EUR<br>AOV: %{customdata[1]:,.0f} EUR<br>YoY vs 2024: %{customdata[2]:.2f} %<extra></extra>",
))

fig_seasonality.update_layout(xaxis_title="Booking Month", yaxis_title="Bookings")
fig_seasonality.update_layout(**CHART_LAYOUT)
fig_seasonality.update_yaxes(tickformat=",.0f")

os.makedirs("output", exist_ok=True)
fig_seasonality.write_html("output/seasonality.html", full_html=False, include_plotlyjs=False)

fig_seasonality.show()

# Answers to questions

## 1. Which dimension(s) have the biggest impact on AOV (Average Order Value)?
Based on Eta-squared, which shows how much of the variability in the data is explained by a given factor, the biggest impact on AOV comes from Passenger Group (39 %) and Destination Type (30 %). Market (4 %), Channel (3 %) and Departure Season (0 %) play almost no role.

Within Passenger Group, Groups record the highest AOV, followed by Families, then couples, with Singles lowest. This gap mostly reflects order size rather than price, since AOV is value per booking and the source data holds no passenger counts, so a group order simply bundles more travellers into one transaction.

Within Destination Type, Exotic leads by a wide margin ahead of Sun and Beach, with Hotel Only lowest. Unlike the passenger effect this is a genuine product price difference and it holds regardless of who is travelling.

## 2. What are the main GBV (Gross Booking Value) drivers?
GBV is driven by volume, not by price. The bulk of the value sits in the Czech and Polish markets (74 % together) and with Families (48 %). Sun and Beach trips and Summer departures dominate as well, but only because that is where the bookings already are, so AOV plays no role in that split.

Within Market, Czechia and Poland deliver most of the value, while Slovakia and Hungary contribute slightly more GBV than their share of bookings thanks to a higher order value.Within Passenger Group, Families and Groups turn 47 % of bookings into 61 % of GBV, while Singles drop from 4 % to 2 %. This is the only dimension where order value visibly shifts the mix.

## 3. What are the year-over-year dynamics of „First Minute season”
First Minute is difficult to compare year on year, because the data contains only one complete season cycle. Yaer 2024 is only full reference year against an unfinished 2025, which has no December at all and only a partial November. Those two months matter more than any others here, since they carry the highest First Minute share of the year.

Compared on the same January to October window, First Minute is stable at 24,1 % of bookings in 2025 against 24,2 % in 2024, and the same holds in both seasons. What sets it apart is value rather than volume, since those bookings carry an average order value of 2 419 € against 2 009 € for the rest, a premium that is unchanged year on year.

In [56]:
### Determining first minute bookings ###

# Difference between departure year and booking year
year_diff = clear_data["departure_year"] - clear_data["booking_year"]

# Row division by season
is_summer = clear_data["departure_season"] == "Summer"
is_winter = clear_data["departure_season"] == "Winter"

# Determining First Minute in the summer season
summer_fm = (year_diff >= 1) | ((year_diff == 0) & clear_data["booking_month"].isin(["Jan", "Feb", "Mar"]))
# Determining First Minute in the winter season
winter_fm = (year_diff >= 2) | ((year_diff == 1) & clear_data["booking_month"].isin(["Aug", "Sep", "Oct", "Nov"]))

# Inserting the final column
clear_data["first_minute"] = (is_summer & summer_fm) | (is_winter & winter_fm)
clear_data.head(10)

,market,departure_season,departure_year,booking_month,booking_year,passenger_group,destination_type,channel,bookings,aov,gbv_eur,value_x_weight,first_minute
0,PL,Winter,2026,Aug,2025,Young Couple (<30),Exotic,Offline,1,46129.000000,46129,46129.000000,True
1,PL,Winter,2023,Nov,2024,Families,Exotic,Online,1,40778.000000,40778,40778.000000,False
2,PL,Winter,2026,May,2025,Groups,Exotic,Offline,4,32059.250000,128237,128237.000000,False
3,HU,Winter,2024,Aug,2023,Families,Exotic,Online,1,30911.000000,30911,30911.000000,True
4,CZ,Winter,2024,Jun,2024,Groups,Other,Offline,1,27951.000000,27951,27951.000000,False
5,SK,Winter,2024,May,2024,Groups,Exotic,Offline,1,25710.000000,25710,25710.000000,False
6,SK,Winter,2025,Jun,2024,Groups,Exotic,Offline,1,25695.000000,25695,25695.000000,False
7,SK,Summer,2025,Sep,2024,Families,Exotic,Offline,1,25245.000000,25245,25245.000000,True
8,PL,Summer,2024,Nov,2024,Groups,Sun & Beach,Offline,1,24454.000000,24454,24454.000000,False
9,PL,Summer,2025,Oct,2024,Groups,Exotic,Offline,3,23536.666016,70610,70609.998047,True


In [57]:
seasons = ["Summer", "Winter"]
pct_2024, pct_2025 = [], []
for season in seasons:
    sub24 = clear_data[(clear_data.departure_season == season) & (clear_data.booking_year == 2024)]
    sub25 = clear_data[(clear_data.departure_season == season) & (clear_data.booking_year == 2025)]
    pct_2024.append(sub24[sub24.first_minute].bookings.sum() / sub24.bookings.sum() * 100)
    pct_2025.append(sub25[sub25.first_minute].bookings.sum() / sub25.bookings.sum() * 100)

x = np.arange(len(seasons))
width = 0.35
fig_first_minute_yoy_season = go.Figure()
fig_first_minute_yoy_season.add_trace(go.Bar(x=x - width / 2, y=pct_2024, width=width, name="2024", marker_color=blue_dark,
    text=[f"{format_eu(v, 1)} %" for v in pct_2024], textposition="outside",
    customdata=seasons,
    hovertemplate="<b>%{customdata} 2024</b><br>First Minute share: %{y:.1f} %<extra></extra>"))
fig_first_minute_yoy_season.add_trace(go.Bar(x=x + width / 2, y=pct_2025, width=width, name="2025", marker_color=sea_blue,
    text=[f"{format_eu(v, 1)} %" for v in pct_2025], textposition="outside",
    customdata=seasons,
    hovertemplate="<b>%{customdata} 2025</b><br>First Minute share: %{y:.1f} %<extra></extra>"))

fig_first_minute_yoy_season.update_layout(title="First Minute Share YoY by Season (2024 vs 2025)", yaxis_title="% First Minute (Bookings)")
fig_first_minute_yoy_season.update_layout(**CHART_LAYOUT)
fig_first_minute_yoy_season.update_xaxes(tickmode="array", tickvals=x, ticktext=seasons)

os.makedirs("output", exist_ok=True)
fig_first_minute_yoy_season.write_html("output/first_minute_yoy_season.html", full_html=False, include_plotlyjs=False)

fig_first_minute_yoy_season.show()

## 4. Which Market do you think Invia has potential to grow further?
Poland has the clearest growth potential. It is the least developed of the four markets relative to its size, with 2,8 bookings per thousand inhabitants against 13,4 in Czechia and 8,7 in Slovakia, yet it has by far the largest population to draw on. Poland is already converting that headroom into results, growing 11,8 % in GBV and 6,7 % in bookings against a company average of 9,1 %.

On the other hand Poland carries the lowest average order value in the portfolio, so added volume converts into GBV less efficiently. Hungary offers comparable room but no momentum, Slovakia strong momentum but little room left, and Czechia grows on price rather than volume.